# Pair Programming Report
## Statistical Classification Log-Loss (Cross-Entropy) with Logistic Regression from Scratch
### Use case: Hours Studied vs. Pass / Fail

**Workshop:** Logistic Regression Classifier Workshop  

**Team A (us):**  

| Member | Student ID |
|---|---|
| Sultan | 9114837 |
| Antonio | 9072844 |
| Eche | 9078881 |
| John | 9115726 |

**Our repo:** (https://github.com/kinghighshow/LogisticRegressionClassifierWorkshop.git)  
**Team B (peer group):** _add names_  
**Team B's repo:** _add link_  

---  

### Roadmap

| # | Section | Rubric area |
|---|---|---|
| 1 | Setup and the data | Reproducibility |
| 2 | Mathematical foundation: model, sigmoid, cross-entropy, gradient | Mathematical Foundation |
| 3 | Sigmoid in code | Implementation |
| 4 | Log-Loss for one student, then for the whole dataset | Implementation |
| 5 | Logistic regression from scratch (NumPy class) | Implementation |
| 6 | Training with gradient descent: loss going down | Gradient Descent and Results |
| 7 | Results: probabilities, classes, confusion matrix | Gradient Descent and Results |
| 8 | Why Log-Loss tells us more than accuracy | Results |
| 9 | Check against scikit-learn (verification only) | Results |
| 10 | Predictions for new students | Results |
| 11 | Talking points + who presents what | Pair Programming |
| 12 | Session notes with Team B | Pair Programming |

**scikit-learn is used only in Section 9, to check our numbers. The model itself is built with NumPy.**  

## 1. Setup and the data

**Libraries**  
- **NumPy**: all the math (vectors, matrices, `exp`, `log`)  
- **pandas**: tables for showing results  
- **Plotly**: interactive charts (hover over points to see values)  

**To reproduce:** `pip install -r requirements.txt`, then *Kernel → Restart & Run All*. The data is fixed and nothing is random, so every run gives the same numbers.  

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

FAIL_COLOR = "#E4572E"
PASS_COLOR = "#2E86AB"
CURVE_COLOR = "#6A4C93"

Each row is one student.  
- **`hours`** = hours studied (the **feature**, $x$)  
- **`passed`** = real outcome (the **label**, $y$): `1` = Pass, `0` = Fail  

This is **binary classification**: every answer is one of two classes.  

In [2]:
hours = np.array([0.5, 0.75, 1, 1.25, 1.5, 1.75, 2, 2.25, 2.5, 2.75,
                  3, 3.25, 3.5, 3.75, 4, 4.25, 4.5, 4.75, 5])
passed = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
                   1, 0, 1, 1, 1, 1, 1, 1, 1])

df = pd.DataFrame({"hours": hours, "passed": passed})
print(f"Students: {len(df)}  |  Passed: {passed.sum()}  |  Failed: {(passed == 0).sum()}")
df.T

Students: 19  |  Passed: 9  |  Failed: 10


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
hours,0.5,0.75,1.0,1.25,1.5,1.75,2.0,2.25,2.5,2.75,3.0,3.25,3.5,3.75,4.0,4.25,4.5,4.75,5.0
passed,0.0,0.00,0.0,0.00,0.0,0.00,0.0,0.00,1.0,0.00,1.0,0.00,1.0,1.00,1.0,1.00,1.0,1.00,1.0


**Reading the chart below**  
- **x-axis**: hours studied, **y-axis**: outcome (0 = Fail, 1 = Pass)  
- **Red** = failed, **blue** = passed  
- **Shaded overlap zone (≈ 2.5 to 3.25 h)**: passes and fails mix, so no single cut-off is perfect. The model must work with *probabilities*.  

In [3]:
fig = go.Figure()
for label, name, color in [(0, "Fail", FAIL_COLOR), (1, "Pass", PASS_COLOR)]:
    sub = df[df.passed == label]
    fig.add_trace(go.Scatter(x=sub.hours, y=sub.passed, mode="markers", name=name,
                             marker=dict(size=13, color=color, line=dict(width=1, color="white"))))
fig.add_vrect(x0=2.4, x1=3.35, fillcolor="gray", opacity=0.12, line_width=0,
              annotation_text="overlap zone", annotation_position="top left")
fig.update_layout(title="Hours Studied vs. Exam Outcome", template="plotly_white",
                  xaxis_title="Hours studied", yaxis_title="Outcome (0 = Fail, 1 = Pass)",
                  yaxis=dict(tickvals=[0, 1]), height=420)
fig.show()

## 2. Mathematical foundation

### 2.1 The linear score

Logistic regression starts with a straight line:  

$$ z = w_0 + w_1 x $$

- $w_0$ = **intercept** (bias), $w_1$ = **weight** for hours, $x$ = hours studied  

**Vector form (what the code uses).** Add a column of 1s to the data so the intercept is treated like any other weight:  

$$ \mathbf{X} = \begin{bmatrix} 1 & x_1 \\ 1 & x_2 \\ \vdots & \vdots \\ 1 & x_N \end{bmatrix}, \qquad \mathbf{w} = \begin{bmatrix} w_0 \\ w_1 \end{bmatrix}, \qquad \mathbf{z} = \mathbf{X}\mathbf{w} $$

One matrix multiplication gives the score for **all students at once**. This is what "vectorized" means: no Python loop over students.  

### 2.2 The sigmoid: turning a score into a probability

$z$ can be any number, so we squeeze it into the range 0 to 1:  

$$ p = \sigma(z) = \frac{1}{1 + e^{-z}} $$

| z | σ(z) | meaning |
|---|---|---|
| very negative | ≈ 0 | very likely **Fail** |
| 0 | 0.5 | 50/50 |
| very positive | ≈ 1 | very likely **Pass** |

**Its derivative** (needed for gradient descent). Write $\sigma(z) = (1 + e^{-z})^{-1}$ and differentiate:  

$$ \sigma'(z) = \frac{e^{-z}}{(1 + e^{-z})^2} = \underbrace{\frac{1}{1 + e^{-z}}}_{\sigma(z)} \cdot \underbrace{\frac{e^{-z}}{1 + e^{-z}}}_{1 - \sigma(z)} $$

$$ \boxed{\;\sigma'(z) = \sigma(z)\,\big(1 - \sigma(z)\big) = p\,(1 - p)\;} $$

### 2.3 Binary cross-entropy (Log-Loss): where the formula comes from

Each student either passes or fails, so the model's probability of the **real** outcome is:  

$$ P(y \mid x) = p^{\,y}\,(1 - p)^{\,1 - y} $$

(If $y = 1$ this is $p$. If $y = 0$ it is $1 - p$.)  

**Likelihood** = how probable the model finds **all** the real outcomes together (students are independent, so multiply):  

$$ \mathcal{L}(\mathbf{w}) = \prod_{i=1}^{N} p_i^{\,y_i}\,(1 - p_i)^{\,1 - y_i} $$

A good model makes this large. Products of many small numbers are awkward, so take the log (turns products into sums):  

$$ \log \mathcal{L}(\mathbf{w}) = \sum_{i=1}^{N} \big[\, y_i \log p_i + (1 - y_i)\log(1 - p_i) \,\big] $$

We want something to **minimize**, so flip the sign and average over $N$ students. That is the **binary cross-entropy / Log-Loss**:  

$$ \boxed{\;J(\mathbf{w}) = -\frac{1}{N} \sum_{i=1}^{N} \big[\, y_i \log p_i + (1 - y_i)\log(1 - p_i) \,\big]\;} $$

**Minimizing Log-Loss = maximizing likelihood.** Same goal, written two ways.  

### 2.4 The gradient: which way is downhill?

For **one** student, $\ell = -\big[\,y \log p + (1 - y)\log(1 - p)\,\big]$, with $p = \sigma(z)$ and $z = w_0 + w_1 x$.  

Use the **chain rule** in three small pieces:  

**(a)** How the loss changes with $p$:  
$$ \frac{\partial \ell}{\partial p} = -\frac{y}{p} + \frac{1 - y}{1 - p} = \frac{p - y}{p\,(1 - p)} $$

**(b)** How $p$ changes with $z$ (from 2.2):  
$$ \frac{\partial p}{\partial z} = p\,(1 - p) $$

**(c)** How $z$ changes with each weight:  
$$ \frac{\partial z}{\partial w_0} = 1, \qquad \frac{\partial z}{\partial w_1} = x $$

**Multiply them.** The $p(1-p)$ terms cancel, leaving something very simple:  

$$ \frac{\partial \ell}{\partial w_0} = (p - y), \qquad \frac{\partial \ell}{\partial w_1} = (p - y)\,x $$

$p - y$ is just the **error** (prediction minus truth). Averaging over all students and writing it in vector form:  

$$ \boxed{\;\nabla J(\mathbf{w}) = \frac{1}{N}\,\mathbf{X}^{\top}(\mathbf{p} - \mathbf{y})\;} $$

### 2.5 Gradient descent update

The gradient points **uphill**, so we step the opposite way:  

$$ \boxed{\;\mathbf{w} \leftarrow \mathbf{w} - \alpha\,\nabla J(\mathbf{w})\;} $$

- $\alpha$ = **learning rate** (step size)  
- Repeat for many **epochs** (passes over the data) until the loss stops dropping  

**Full algorithm**  
1. Add a column of 1s to get $\mathbf{X}$; start with $\mathbf{w} = [0, 0]$  
2. $\mathbf{p} = \sigma(\mathbf{X}\mathbf{w})$  
3. Record the loss $J(\mathbf{w})$  
4. $\nabla = \frac{1}{N}\mathbf{X}^{\top}(\mathbf{p} - \mathbf{y})$  
5. $\mathbf{w} \leftarrow \mathbf{w} - \alpha \nabla$  
6. Go back to step 2 until done  
7. Predict: class = 1 if $p \ge 0.5$, else 0  

**Decision boundary:** where $p = 0.5$, i.e. $z = 0$, so $x_{boundary} = -\,w_0 / w_1$.  

## 3. Sigmoid in code

`np.clip(z, -500, 500)` stops `np.exp` from overflowing on huge numbers. Inside the normal range it changes nothing.  

In [4]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

for z in [-5, -1, 0, 1, 5]:
    print(f"sigmoid({z:>2}) = {sigmoid(z):.3f}")

sigmoid(-5) = 0.007
sigmoid(-1) = 0.269
sigmoid( 0) = 0.500
sigmoid( 1) = 0.731
sigmoid( 5) = 0.993


**Reading the chart below**  
- **Purple S-curve**: sigmoid output for each `z`  
- **Orange curve**: its derivative $p(1-p)$. It peaks at **0.25** when $z = 0$, where the model is least sure, and is near 0 at the ends  
- **Dashed line**: the 0.5 **decision threshold**  

In [5]:
z = np.linspace(-8, 8, 200)
p = sigmoid(z)
fig = go.Figure()
fig.add_trace(go.Scatter(x=z, y=p, mode="lines", name="σ(z)", line=dict(color=CURVE_COLOR, width=4)))
fig.add_trace(go.Scatter(x=z, y=p * (1 - p), mode="lines", name="σ'(z) = p(1 − p)",
                         line=dict(color="#F18F01", width=3, dash="dot")))
fig.add_hline(y=0.5, line_dash="dash", line_color="gray", annotation_text="threshold = 0.5")
fig.update_layout(title="Sigmoid and Its Derivative", template="plotly_white",
                  xaxis_title="z  (linear score)", yaxis_title="Value", height=420)
fig.show()

## 4. Log-Loss in code

### 4.1 One student

**Why clip?** `log(0)` is minus infinity. `np.clip` keeps `p` between `1e-15` and `1 − 1e-15` so the math never breaks.  

In [6]:
EPS = 1e-15

def single_log_loss(y_true, p):
    p = np.clip(p, EPS, 1 - EPS)
    return -(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

cases = pd.DataFrame({
    "situation": ["Passed, model said 95% pass", "Passed, model said 10% pass",
                  "Failed, model said 5% pass",  "Failed, model said 90% pass"],
    "y": [1, 1, 0, 0],
    "p": [0.95, 0.10, 0.05, 0.90],
})
cases["loss"] = single_log_loss(cases.y, cases.p).round(3)
cases

,situation,y,p,loss
0,"Passed, model said 95% pass",1,0.95,0.051
1,"Passed, model said 10% pass",1,0.10,2.303
2,"Failed, model said 5% pass",0,0.05,0.051
3,"Failed, model said 90% pass",0,0.90,2.303


Confident and right costs **≈ 0.05**. Confident and wrong costs **≈ 2.3**, about **45× more**.  

**Reading the chart below**  
- **Left (green)**: student **passed**. Loss → 0 as $p$ → 1, and shoots up as $p$ → 0  
- **Right (orange)**: student **failed**. Mirror image  
- The steep wall = **confident mistakes are punished hardest**  

In [7]:
p = np.linspace(0.001, 0.999, 300)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Student PASSED  (y = 1)", "Student FAILED  (y = 0)"))
fig.add_trace(go.Scatter(x=p, y=-np.log(p), mode="lines", line=dict(color="#2CA58D", width=4)), row=1, col=1)
fig.add_trace(go.Scatter(x=p, y=-np.log(1 - p), mode="lines", line=dict(color="#F18F01", width=4)), row=1, col=2)
fig.add_annotation(x=0.18, y=4, text="confident & WRONG", showarrow=False, row=1, col=1)
fig.add_annotation(x=0.82, y=4, text="confident & WRONG", showarrow=False, row=1, col=2)
fig.update_xaxes(title_text="Predicted probability of passing (p)")
fig.update_yaxes(title_text="Log-Loss", col=1)
fig.update_layout(title="Log-Loss Penalty Curves", template="plotly_white", height=430, showlegend=False)
fig.show()

### 4.2 The whole dataset (vectorized)

This is the boxed formula from 2.3. NumPy works on the full arrays at once, so there is no loop.  

In [8]:
def log_loss(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.clip(np.asarray(y_prob), EPS, 1 - EPS)
    return -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))

**Sanity check with the instructor's 4-student example** (expected average ≈ **0.471**):  

In [9]:
example_y = np.array([0, 1, 1, 0])
example_p = np.array([0.12, 0.45, 0.96, 0.60])
print(f"Average Log-Loss = {log_loss(example_y, example_p):.3f}")

Average Log-Loss = 0.471


## 5. Logistic regression from scratch

Everything from Section 2 in one class.  

| Method | What it does | Formula |
|---|---|---|
| `_add_bias(x)` | adds the column of 1s | builds $\mathbf{X}$ |
| `fit(x, y)` | runs gradient descent, saves the loss each epoch | $\mathbf{w} \leftarrow \mathbf{w} - \frac{\alpha}{N}\mathbf{X}^{\top}(\mathbf{p} - \mathbf{y})$ |
| `predict_proba(x)` | probability of passing | $\sigma(\mathbf{X}\mathbf{w})$ |
| `predict(x)` | Pass/Fail using the threshold | $p \ge 0.5$ |
| `decision_boundary()` | hours where $p = 0.5$ | $-w_0 / w_1$ |

In [10]:
class LogisticRegressionScratch:
    def __init__(self, learning_rate=0.5, epochs=5000, threshold=0.5):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.threshold = threshold
        self.weights = None
        self.loss_history = []

    def _add_bias(self, x):
        x = np.asarray(x, dtype=float).reshape(-1, 1)
        return np.hstack([np.ones((x.shape[0], 1)), x])

    def fit(self, x, y):
        X = self._add_bias(x)
        y = np.asarray(y, dtype=float)
        self.weights = np.zeros(X.shape[1])
        self.loss_history = []
        for _ in range(self.epochs):
            p = sigmoid(X @ self.weights)
            self.loss_history.append(log_loss(y, p))
            gradient = X.T @ (p - y) / len(y)
            self.weights -= self.learning_rate * gradient
        return self

    def predict_proba(self, x):
        return sigmoid(self._add_bias(x) @ self.weights)

    def predict(self, x):
        return (self.predict_proba(x) >= self.threshold).astype(int)

    def decision_boundary(self):
        w0, w1 = self.weights
        return -w0 / w1

## 6. Training with gradient descent

`learning_rate = 0.5`, `epochs = 5000`. Weights start at $[0, 0]$, so every student starts at $p = 0.5$ and the starting loss is $-\log(0.5) \approx 0.693$.  

In [11]:
model = LogisticRegressionScratch(learning_rate=0.5, epochs=5000)
model.fit(hours, passed)

w0, w1 = model.weights
boundary = model.decision_boundary()
final_loss = model.loss_history[-1]

print(f"Intercept w0      = {w0:.3f}")
print(f"Weight    w1      = {w1:.3f}")
print(f"Decision boundary = {boundary:.2f} hours")
print(f"Log-Loss: start = {model.loss_history[0]:.3f}  →  end = {final_loss:.4f}")

Intercept w0      = -8.535
Weight    w1      = 2.968
Decision boundary = 2.88 hours
Log-Loss: start = 0.693  →  end = 0.2321


**Reading the chart below**  
- **x-axis**: epoch, **y-axis**: Log-Loss (log scale on x so the early drop is visible)  
- **Starts at 0.693** (coin flip), **drops fast**, then **flattens**: the weights have converged  
- **Dotted line**: coin-flip baseline  

In [12]:
fig = go.Figure(go.Scatter(x=np.arange(1, model.epochs + 1), y=model.loss_history, mode="lines",
                           line=dict(color=CURVE_COLOR, width=3)))
fig.add_hline(y=np.log(2), line_dash="dot", line_color="gray", annotation_text="coin flip (0.693)")
fig.update_layout(title="Gradient Descent: Log-Loss per Epoch", template="plotly_white",
                  xaxis_title="Epoch", yaxis_title="Log-Loss", xaxis_type="log", height=420)
fig.show()

### 6.1 Effect of the learning rate

Same data, same 5000 epochs, three step sizes.  

**Reading the chart below**  
- **α = 0.01 (small steps)**: loss is still high at the end, training is too slow  
- **α = 0.1**: gets there, but needs most of the epochs  
- **α = 0.5**: converges quickly. This is the value we use  

In [13]:
rates = {0.01: "#9E9E9E", 0.1: "#F18F01", 0.5: CURVE_COLOR}
fig = go.Figure()
summary = []
for lr, color in rates.items():
    m = LogisticRegressionScratch(learning_rate=lr, epochs=5000).fit(hours, passed)
    fig.add_trace(go.Scatter(y=m.loss_history, mode="lines", name=f"α = {lr}", line=dict(color=color, width=3)))
    summary.append({"learning rate": lr, "loss after 1000 epochs": m.loss_history[999],
                    "final loss": m.loss_history[-1]})
fig.update_layout(title="Learning Rate Comparison", template="plotly_white",
                  xaxis_title="Epoch", yaxis_title="Log-Loss", height=420)
fig.show()
pd.DataFrame(summary).round(4)

,learning rate,loss after 1000 epochs,final loss
0,0.01,0.4742,0.3027
1,0.10,0.2627,0.2332
2,0.50,0.2332,0.2321


## 7. Results: probabilities and classes

For each student: the predicted **probability of passing**, the **class** after the 0.5 threshold, and that student's **Log-Loss**.  

In [14]:
p_pass = model.predict_proba(hours)
predicted = model.predict(hours)

results = pd.DataFrame({
    "hours": hours,
    "actual": passed,
    "P(pass)": p_pass.round(3),
    "predicted": predicted,
    "correct?": np.where(predicted == passed, "yes", "NO"),
    "loss": single_log_loss(passed, p_pass).round(3),
})
results

,hours,actual,P(pass),predicted,correct?,loss
0,0.50,0,0.001,0,yes,0.001
1,0.75,0,0.002,0,yes,0.002
2,1.00,0,0.004,0,yes,0.004
3,1.25,0,0.008,0,yes,0.008
4,1.50,0,0.017,0,yes,0.017
5,1.75,0,0.034,0,yes,0.035
6,2.00,0,0.069,0,yes,0.072
7,2.25,0,0.135,0,yes,0.145
8,2.50,1,0.247,0,NO,1.398
9,2.75,0,0.408,0,yes,0.525


**Reading the chart below**  
- **Dots**: real students (red = failed, blue = passed)  
- **Purple S-curve**: our model's probability of passing for any number of hours  
- **Green dashed line**: decision boundary. Right of it → predicted Pass  
- **Circled dots**: the students our model got **wrong**  

In [15]:
x_line = np.linspace(0, 6, 300)
wrong = results[results["correct?"] == "NO"]

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_line, y=model.predict_proba(x_line), mode="lines", name="P(pass)",
                         line=dict(color=CURVE_COLOR, width=4)))
for label, name, color in [(0, "Fail", FAIL_COLOR), (1, "Pass", PASS_COLOR)]:
    sub = results[results.actual == label]
    fig.add_trace(go.Scatter(x=sub.hours, y=sub.actual, mode="markers", name=name,
                             marker=dict(size=12, color=color)))
fig.add_trace(go.Scatter(x=wrong.hours, y=wrong.actual, mode="markers", name="Misclassified",
                         marker=dict(size=24, color="rgba(0,0,0,0)", line=dict(width=2.5, color="black"))))
fig.add_vline(x=boundary, line_dash="dash", line_color="green", annotation_text=f"boundary = {boundary:.2f} h")
fig.add_hline(y=0.5, line_dash="dot", line_color="gray")
fig.update_layout(title="Our Model: Probability of Passing", template="plotly_white",
                  xaxis_title="Hours studied", yaxis_title="Probability of passing", height=450)
fig.show()

### 7.1 Confusion matrix (computed with NumPy)

| | Predicted Fail | Predicted Pass |
|---|---|---|
| **Actually Failed** | **TN**: correctly said Fail | **FP**: said Pass, but failed |
| **Actually Passed** | **FN**: said Fail, but passed | **TP**: correctly said Pass |

- **Accuracy** = (TP + TN) / all  
- **Precision** = TP / (TP + FP): of those we said would pass, how many did  
- **Recall** = TP / (TP + FN): of those who passed, how many we caught  

**Reading the chart below**  
- **Rows** = real outcome, **columns** = prediction  
- **Diagonal** = correct, **off-diagonal** = mistakes, **darker** = more students  

In [16]:
tn = np.sum((passed == 0) & (predicted == 0))
fp = np.sum((passed == 0) & (predicted == 1))
fn = np.sum((passed == 1) & (predicted == 0))
tp = np.sum((passed == 1) & (predicted == 1))
cm = np.array([[tn, fp], [fn, tp]])

labels = ["Fail (0)", "Pass (1)"]
names = [["TN", "FP"], ["FN", "TP"]]
text = [[f"{names[i][j]}<br><b>{cm[i][j]}</b>" for j in range(2)] for i in range(2)]

fig = go.Figure(go.Heatmap(z=cm, x=labels, y=labels, text=text, texttemplate="%{text}",
                           textfont=dict(size=22), colorscale="Blues", showscale=False,
                           hovertemplate="Actual: %{y}<br>Predicted: %{x}<br>Students: %{z}<extra></extra>"))
fig.update_layout(title="Confusion Matrix", template="plotly_white", xaxis_title="Predicted",
                  yaxis_title="Actual", yaxis=dict(autorange="reversed"), height=450, width=500)
fig.show()

accuracy = (tp + tn) / cm.sum()
precision = tp / (tp + fp)
recall = tp / (tp + fn)
print(f"Accuracy {accuracy:.2%}  |  Precision {precision:.2%}  |  Recall {recall:.2%}")

Accuracy 89.47%  |  Precision 88.89%  |  Recall 88.89%


### 7.2 Which students create the loss?

**Reading the chart below**  
- One **bar** per student, **colour** = real outcome  
- **Tallest bars**: the two misclassified students (2.5 h passed, 3.25 h failed)  
- **Medium bars**: correct but close to the boundary, where the model was unsure  
- **Near-zero bars**: far from the boundary, confident **and** right  
- **Dashed line**: the average = final Log-Loss  

In [17]:
colors = [PASS_COLOR if v == 1 else FAIL_COLOR for v in results.actual]
fig = go.Figure(go.Bar(x=results.hours.astype(str), y=results.loss, marker_color=colors,
                       customdata=results["P(pass)"],
                       hovertemplate="hours=%{x}<br>loss=%{y}<br>P(pass)=%{customdata}<extra></extra>"))
fig.add_hline(y=final_loss, line_dash="dash", line_color="black", annotation_text=f"average = {final_loss:.3f}")
fig.update_layout(title="Log-Loss per Student", template="plotly_white",
                  xaxis_title="Hours studied (one bar per student)", yaxis_title="Log-Loss", height=430)
fig.show()

## 8. Why Log-Loss tells us more than accuracy

| Model | How it predicts |
|---|---|
| **Coin flip** | always 50% (0.5 counts as Pass, so it only gets the 9 passers right) |
| **Our model** | trained weights |
| **Over-confident** | same boundary, but weights × 4, so probabilities are pushed to almost 0 or 1 |

Our model and the over-confident one make **identical Pass/Fail decisions**, so accuracy is identical. Only Log-Loss separates them.  

In [18]:
X_all = model._add_bias(hours)
p_coin = np.full_like(p_pass, 0.5)
p_over = sigmoid(X_all @ (4 * model.weights))

compare = pd.DataFrame({
    "model": ["Coin flip", "Our model", "Over-confident"],
    "accuracy": [np.mean((p >= 0.5).astype(int) == passed) for p in [p_coin, p_pass, p_over]],
    "log_loss": [log_loss(passed, p) for p in [p_coin, p_pass, p_over]],
}).round(3)
compare

,model,accuracy,log_loss
0,Coin flip,0.474,0.693
1,Our model,0.895,0.232
2,Over-confident,0.895,0.491


**Reading the chart below**  
- **Left**: accuracy. Our model and the over-confident model look equal  
- **Right**: Log-Loss. The over-confident model is clearly **worse**: its two mistakes were made with near-certainty  
- **Lesson:** Log-Loss rewards a model that is honest about its uncertainty  

In [22]:
bar_colors = ["#9E9E9E", CURVE_COLOR, "#D1495B"]
fig = make_subplots(rows=1, cols=2, subplot_titles=("Accuracy (higher is better)", "Log-Loss (lower is better)"))
fig.add_trace(go.Bar(x=compare.model, y=compare.accuracy, marker_color=bar_colors,
                     text=compare.accuracy, textposition="outside"), row=1, col=1)
fig.add_trace(go.Bar(x=compare.model, y=compare.log_loss, marker_color=bar_colors,
                     text=compare.log_loss, textposition="outside"), row=1, col=2)
fig.update_layout(title="Same Accuracy, Different Log-Loss", template="plotly_white",
                  height=630, showlegend=False)
fig.show()

## 9. Check against scikit-learn (verification only)

scikit-learn is **not** used to build our model. We use it here only to confirm our math.  

- `LogisticRegression(C=np.inf)` turns off scikit-learn's default penalty (regularization), so it solves exactly the same problem as our gradient descent  
- `log_loss` from scikit-learn should match our `log_loss` function  

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss as sk_log_loss

sk_model = LogisticRegression(C=np.inf, max_iter=10000).fit(hours.reshape(-1, 1), passed)
sk_p = sk_model.predict_proba(hours.reshape(-1, 1))[:, 1]

pd.DataFrame({
    "": ["w0 (intercept)", "w1 (weight)", "Log-Loss"],
    "Our NumPy model": [w0, w1, log_loss(passed, p_pass)],
    "scikit-learn": [sk_model.intercept_[0], sk_model.coef_[0][0], sk_log_loss(passed, sk_p)],
}).round(4)

,,Our NumPy model,scikit-learn
0,w0 (intercept),-8.5348,-8.5400
1,w1 (weight),2.9684,2.9703
2,Log-Loss,0.2321,0.2321


The weights agree to about two decimal places and the Log-Loss matches to four. Our from-scratch gradient descent finds the same answer as the library.  

## 10. Predictions for new students

- **P(pass)** and **prediction** from our model  
- **Loss if passes / if fails**: how surprised the model would be by each real outcome  

In [21]:
new_hours = np.array([1.0, 2.6, 3.0, 4.0])
new_p = model.predict_proba(new_hours)

pd.DataFrame({
    "hours": new_hours,
    "P(pass)": new_p.round(3),
    "prediction": np.where(model.predict(new_hours) == 1, "Pass", "Fail"),
    "loss if passes": single_log_loss(1, new_p).round(3),
    "loss if fails": single_log_loss(0, new_p).round(3),
})

,hours,P(pass),prediction,loss if passes,loss if fails
0,1.0,0.004,Fail,5.570,0.004
1,2.6,0.306,Fail,1.183,0.366
2,3.0,0.592,Pass,0.525,0.895
3,4.0,0.966,Pass,0.035,3.374


Near the boundary (2.6 h, 3 h) both possible losses are moderate: the model is unsure. Far from it (1 h, 4 h) one loss is tiny and the other is large: the model has committed.  

## 11. Talking Points (Team A)

**1. Cross-entropy comes from probability, and it penalizes confident wrong predictions.**

Log-Loss measures how well a model predicts the correct outcome. If a model predicts a 10% chance of passing when a student actually passes, it receives a high penalty of about 2.3. A confident and correct prediction gets a much smaller penalty of about 0.05. This means being confidently wrong is worse than being slightly wrong. Two models can have the same accuracy but different Log-Loss scores because one makes better probability predictions.

**2. The gradient simplifies to "error × input", making the code faster and simpler.**

Gradient descent improves the model by checking the difference between its predictions and the actual results, then adjusting its weights. The formula simplifies to prediction error multiplied by the input data. This allows us to update the model for all students at once without using a loop. In our code, this is done in one line: `X.T @ (p - y) / len(y)`. The column of 1s in our input data also allows the model to update its bias.

**3. Gradient descent reduces the loss, and our results match scikit-learn.**

Our model starts with a Log-Loss of about 0.693, which represents a 50/50 prediction. As training continues, the loss decreases, meaning the model is improving. With a learning rate of 0.5, the loss drops below 0.25 within 300 epochs and settles around 0.232 after about 2,000 epochs. With a learning rate of 0.01, learning is slower, and the loss remains around 0.30 after 5,000 epochs. Our model's weights and Log-Loss are close to those from scikit-learn, a popular machine learning library. It achieves 89.5% accuracy, misclassifying only the two students in the overlapping region between the two classes.
### Who presents what

| Part of the walkthrough | Sections | Presenter |
|---|---|---|
| Data and the math (sigmoid, cross-entropy, gradient) | 1, 2 | Sultan |
| Log-Loss code and Talking point 1 | 3, 4, 8 | Antonio |
| The from-scratch class and Talking point 2 | 5 | John |
| Training, results, verification and Talking point 3 | 6, 7, 9, 10 | Eche |

## 12. Session Notes with Team B

### a) Our review of Team B's code (step 4a)
**Questions we asked about their code and their answers:**  
1.  
2.  
3.  

### b) Team B's review of our code (step 4b)
**Questions they asked about our code and how we answered:**  
1.  
2.  
3.  

### c) Our notes and insights on Team B's talking points
-  
-  
-  

### d) Team B's notes and insights on our talking points
-  
-  
-  

### e) What we would change after the review
-  